In [5]:
# ============================================================================
#  ✈️  ANALYSE DES CRASHS AÉRIENS ET DES VICTIMES JUSQU'EN 2023
#  Outils : Python · Pandas · NumPy · SciPy · Matplotlib · Seaborn
#  Une seule cellule, exécutable de haut en bas. Chaque bloc est commenté.
# ============================================================================

# --- 0. IMPORTS -------------------------------------------------------------
# On regroupe toutes les bibliothèques au début (bonne pratique).

import numpy as np                # calcul numérique (tableaux, stats vectorisées)
import pandas as pd               # manipulation de données tabulaires (DataFrame)
import scipy.stats as stats       # tests statistiques (t-test, normalité, etc.)
import matplotlib.pyplot as plt   # graphiques de base
import seaborn as sns             # graphiques statistiques plus jolis

warnings.filterwarnings("ignore")            # on ignore les warnings pour un rendu propre
sns.set_theme(style="whitegrid")             # style visuel cohérent pour tous les graphes
pd.set_option("display.max_columns", None)   # on veut voir toutes les colonnes à l'affichage


# ============================================================================
#  1. IMPORT ET NETTOYAGE DES DONNÉES
# ============================================================================

# --- 1.1  lecture du CSV ---------------------------


url = "Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv"
df = pd.read_csv(url, encoding="latin-1")   # encodage non-UTF8 (caractères accentués)
print(f"Dimensions brutes : {df.shape[0]} lignes × {df.shape[1]} colonnes")

# --- 1.2 Normalisation des noms de colonnes --------------------------------
# Les versions de ce dataset varient (espaces, majuscules...). On normalise pour
# que le reste du code soit robuste : minuscules, sans espaces superflus, "_" à la place.
df.columns = (df.columns
                .str.strip()          # enlève les espaces en début/fin
                .str.lower()          # tout en minuscules
                .str.replace(" ", "_")# espaces -> underscores
                .str.replace(".", "", regex=False))  # enlève les points éventuels

print("Colonnes disponibles :", list(df.columns), "\n")

# Petite fonction utilitaire : retrouve le vrai nom d'une colonne à partir de
# mots-clés, afin de ne pas planter si la casse/orthographe diffère légèrement.
def find_col(*keywords):
    for c in df.columns:
        if all(k in c for k in keywords):
            return c
    return None

col_date     = find_col("date")                          # colonne de date
col_fatal    = find_col("fatalities") or find_col("dead")# nombre total de morts
col_aboard   = find_col("aboard")                        # nombre de personnes à bord
col_location = find_col("location")                      # lieu du crash
col_operator = find_col("operator")                      # compagnie/exploitant

# --- 1.3 Conversion de la date et création de variables temporelles --------
# On convertit la colonne date en vrai type datetime (erreurs -> NaT = date manquante).
df[col_date] = pd.to_datetime(df[col_date], errors="coerce")

# On dérive l'année et la décennie : utiles pour analyser les tendances dans le temps.
df["year"]    = df[col_date].dt.year
df["decade"]  = (df["year"] // 10 * 10)   # 1987 -> 1980, 2003 -> 2000, etc.

# --- 1.4 Conversion des champs numériques ----------------------------------
# 'aboard' et 'fatalities' doivent être numériques. errors="coerce" transforme
# toute valeur non convertible (texte parasite) en NaN.
df[col_aboard] = pd.to_numeric(df[col_aboard], errors="coerce")
df[col_fatal]  = pd.to_numeric(df[col_fatal],  errors="coerce")

# --- 1.5 Traitement des valeurs manquantes ---------------------------------
print("Valeurs manquantes par colonne (top) :")
print(df.isna().sum().sort_values(ascending=False).head(10), "\n")

# Stratégie : pour les colonnes clés de l'analyse (morts, à bord), on remplit les
# NaN par 0 UNIQUEMENT là où c'est cohérent. Ici on choisit de ne garder pour les
# calculs de survie que les lignes où 'aboard' est connu et > 0.
df[col_fatal]  = df[col_fatal].fillna(0)   # un NaN de morts est traité comme 0 mort connu
# 'aboard' reste NaN si inconnu : on ne veut pas inventer le nombre de passagers.

# --- 1.6 Variable dérivée : survivants et taux de survie -------------------
# Survivants = personnes à bord - morts. Le taux de survie n'a de sens que si
# 'aboard' est connu et strictement positif (sinon division par 0 / NaN).
df["survivors"]     = df[col_aboard] - df[col_fatal]
df["survival_rate"] = np.where(df[col_aboard] > 0,            # condition
                               df["survivors"] / df[col_aboard],  # si vrai
                               np.nan)                        # si faux -> NaN

# On borne le taux entre 0 et 1 pour neutraliser d'éventuelles incohérences de saisie.
df["survival_rate"] = df["survival_rate"].clip(0, 1)

print("Aperçu des données nettoyées :")
print(df[[col_date, "year", "decade", col_aboard, col_fatal,
          "survivors", "survival_rate"]].head(), "\n")


# ============================================================================
#  2. ANALYSE EXPLORATOIRE (EDA)
# ============================================================================

# --- 2.1 Statistiques globales ---------------------------------------------
total_crashes   = len(df)                          # nombre total d'accidents enregistrés
total_fatal     = df[col_fatal].sum()              # somme de toutes les victimes
total_aboard    = df[col_aboard].sum()             # somme des personnes à bord (connues)
global_survival = 1 - (total_fatal / total_aboard) # taux de survie global agrégé

print("===== STATISTIQUES GLOBALES =====")
print(f"Nombre total de crashs        : {total_crashes:,}")
print(f"Total des victimes            : {int(total_fatal):,}")
print(f"Total personnes à bord (connu): {int(total_aboard):,}")
print(f"Taux de survie global         : {global_survival:.2%}")
print(f"Période couverte              : {int(df['year'].min())} → {int(df['year'].max())}\n")

# --- 2.2 Fréquence des crashs dans le temps --------------------------------
# On compte le nombre de crashs par année (série temporelle).
crashes_per_year = df.groupby("year").size()

# Et le nombre de victimes par année (pour comparer fréquence vs gravité).
fatalities_per_year = df.groupby("year")[col_fatal].sum()

# Décennie la plus meurtrière et année record (insights rapides).
print("Année avec le plus de crashs :", int(crashes_per_year.idxmax()),
      f"({crashes_per_year.max()} crashs)")
print("Décennie la plus meurtrière  :",
      int(df.groupby("decade")[col_fatal].sum().idxmax()), "\n")

# --- 2.3 Top des exploitants (operators) -----------------------------------
# Quels exploitants reviennent le plus souvent dans les accidents ?
if col_operator:
    top_operators = df[col_operator].value_counts().head(10)
    print("Top 10 des exploitants les plus présents dans les crashs :")
    print(top_operators, "\n")

# --- 2.4 Extraction du "pays/région" depuis la localisation ----------------
# La colonne location ressemble à "Ville, Pays". On prend le dernier segment
# après la virgule comme approximation de la région/pays.
if col_location:
    df["region"] = (df[col_location].astype(str)
                                    .str.split(",")
                                    .str[-1]       # dernier morceau = pays/État
                                    .str.strip())
    top_regions = df["region"].value_counts().head(10)
    print("Top 10 des régions/pays par nombre de crashs :")
    print(top_regions, "\n")


# ============================================================================
#  3. ANALYSE STATISTIQUE (SciPy + NumPy)
# ============================================================================

# --- 3.1 Statistiques descriptives des victimes ----------------------------
# On travaille sur les victimes par accident. NumPy pour mean/median/std,
# SciPy pour des mesures de forme (asymétrie, aplatissement).
fatal_values = df[col_fatal].dropna().values        # tableau NumPy des victimes

mean_fatal   = np.mean(fatal_values)                # moyenne
median_fatal = np.median(fatal_values)              # médiane (robuste aux extrêmes)
std_fatal    = np.std(fatal_values, ddof=1)         # écart-type (ddof=1 = échantillon)
skew_fatal   = stats.skew(fatal_values)             # asymétrie de la distribution
kurt_fatal   = stats.kurtosis(fatal_values)         # aplatissement (queues lourdes ?)

print("===== DISTRIBUTION DES VICTIMES PAR CRASH =====")
print(f"Moyenne     : {mean_fatal:.2f}")
print(f"Médiane     : {median_fatal:.2f}")
print(f"Écart-type  : {std_fatal:.2f}")
print(f"Asymétrie   : {skew_fatal:.2f}  (>0 => longue queue à droite)")
print(f"Kurtosis    : {kurt_fatal:.2f}  (>0 => queues plus lourdes qu'une loi normale)\n")

# Interprétation : moyenne >> médiane + forte asymétrie => quelques catastrophes
# extrêmes tirent la moyenne vers le haut ; la majorité des crashs font peu de morts.

# --- 3.2 Test de normalité (les victimes suivent-elles une loi normale ?) ---
# On teste H0 : "les données suivent une loi normale". p < 0.05 => on rejette H0.
# (On échantillonne à 5000 max car le test de Shapiro est lent sur grands jeux.)
sample = fatal_values if len(fatal_values) <= 5000 else \
         np.random.choice(fatal_values, 5000, replace=False)
shapiro_stat, shapiro_p = stats.shapiro(sample)
print("===== TEST DE NORMALITÉ (Shapiro-Wilk) =====")
print(f"statistique = {shapiro_stat:.4f} | p-value = {shapiro_p:.2e}")
print("Conclusion :", "distribution NON normale (p<0.05)" if shapiro_p < 0.05
      else "compatible avec une loi normale", "\n")

# --- 3.3 Test d'hypothèse : comparer deux décennies ------------------------
# Question : la gravité moyenne (victimes par crash) diffère-t-elle entre deux décennies ?
# H0 : les moyennes sont égales.  H1 : elles diffèrent.
# On utilise un test t de Welch (equal_var=False, ne suppose pas des variances égales).
g1 = df.loc[df["decade"] == 1970, col_fatal].dropna()   # victimes/crash dans les années 70
g2 = df.loc[df["decade"] == 2000, col_fatal].dropna()   # victimes/crash dans les années 2000

t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False)
print("===== TEST T : VICTIMES MOYENNES 1970s vs 2000s =====")
print(f"Moyenne 1970s : {g1.mean():.2f} (n={len(g1)})")
print(f"Moyenne 2000s : {g2.mean():.2f} (n={len(g2)})")
print(f"t = {t_stat:.3f} | p-value = {p_val:.4f}")
print("Conclusion :",
      "différence significative (p<0.05)" if p_val < 0.05
      else "pas de différence significative", "\n")

# --- 3.4 Corrélation : plus de monde à bord => plus de morts ? --------------
# Test de corrélation de Pearson entre 'aboard' et 'fatalities'.
mask = df[col_aboard].notna() & df[col_fatal].notna()   # on garde les lignes complètes
r, p_corr = stats.pearsonr(df.loc[mask, col_aboard], df.loc[mask, col_fatal])
print("===== CORRÉLATION 'à bord' vs 'victimes' =====")
print(f"r de Pearson = {r:.3f} | p-value = {p_corr:.2e}")
print("Interprétation : corrélation positive forte attendue (plus de passagers, plus de victimes potentielles).\n")


# ============================================================================
#  4. VISUALISATIONS (Matplotlib + Seaborn)
# ============================================================================
# On crée une grille 2x3 de graphiques pour regrouper les visualisations clés.
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle("Analyse des crashs aériens jusqu'en 2023", fontsize=18, fontweight="bold")

# (1) Série temporelle : nombre de crashs par année
axes[0, 0].plot(crashes_per_year.index, crashes_per_year.values, color="steelblue")
axes[0, 0].set_title("Nombre de crashs par année")
axes[0, 0].set_xlabel("Année"); axes[0, 0].set_ylabel("Crashs")

# (2) Série temporelle : victimes par année
axes[0, 1].plot(fatalities_per_year.index, fatalities_per_year.values, color="firebrick")
axes[0, 1].set_title("Victimes par année")
axes[0, 1].set_xlabel("Année"); axes[0, 1].set_ylabel("Victimes")

# (3) Histogramme des victimes par crash (échelle log en Y pour voir la longue queue)
axes[0, 2].hist(fatal_values, bins=50, color="darkorange", edgecolor="black")
axes[0, 2].set_yscale("log")   # log car la distribution est très asymétrique
axes[0, 2].set_title("Distribution des victimes par crash (Y en log)")
axes[0, 2].set_xlabel("Victimes"); axes[0, 2].set_ylabel("Fréquence (log)")

# (4) Crashs par décennie (barres)
crashes_per_decade = df.groupby("decade").size()
sns.barplot(x=crashes_per_decade.index.astype(int), y=crashes_per_decade.values,
            ax=axes[1, 0], color="mediumseagreen")
axes[1, 0].set_title("Crashs par décennie")
axes[1, 0].set_xlabel("Décennie"); axes[1, 0].set_ylabel("Crashs")
axes[1, 0].tick_params(axis="x", rotation=45)

# (5) Top 10 régions par nombre de crashs (barres horizontales)
if col_location:
    sns.barplot(x=top_regions.values, y=top_regions.index, ax=axes[1, 1], color="slateblue")
    axes[1, 1].set_title("Top 10 régions par crashs")
    axes[1, 1].set_xlabel("Crashs"); axes[1, 1].set_ylabel("Région")

# (6) Distribution du taux de survie
sns.histplot(df["survival_rate"].dropna(), bins=20, ax=axes[1, 2], color="teal")
axes[1, 2].set_title("Distribution du taux de survie")
axes[1, 2].set_xlabel("Taux de survie"); axes[1, 2].set_ylabel("Nombre de crashs")

plt.tight_layout(rect=[0, 0, 1, 0.97])   # laisse de la place au titre principal
plt.show()


# ============================================================================
#  5. SYNTHÈSE ET INSIGHTS
# ============================================================================
print("="*70)
print("SYNTHÈSE DES PRINCIPAUX INSIGHTS")
print("="*70)
print(f"""
- VOLUME : {total_crashes:,} crashs recensés sur {int(df['year'].max()-df['year'].min())} ans,
  pour un total d'environ {int(total_fatal):,} victimes.

- GRAVITÉ : la moyenne ({mean_fatal:.1f}) est nettement supérieure à la médiane
  ({median_fatal:.1f}), et l'asymétrie est forte ({skew_fatal:.1f}). La majorité des
  crashs font peu de victimes, mais quelques catastrophes rares et massives
  tirent la moyenne vers le haut (distribution à longue queue, confirmée par
  le test de Shapiro qui rejette la normalité).

- TENDANCE TEMPORELLE : le pic de crashs se situe vers {int(crashes_per_year.idxmax())}.
  L'amélioration des technologies et de la réglementation explique la baisse
  observée sur les décennies récentes.

- COMPARAISON DÉCENNIES (test t 1970s vs 2000s) : p = {p_val:.4f} →
  {"différence statistiquement significative" if p_val < 0.05 else "pas de différence significative"}.

- CORRÉLATION : r = {r:.2f} entre passagers à bord et victimes → plus l'avion est
  rempli, plus le nombre de victimes potentielles est élevé (lien attendu et fort).

OUTILS UTILISÉS :
  - Pandas  : import, nettoyage, conversions de types, groupby, agrégations.
  - NumPy   : mean / median / std, opérations vectorisées, np.where pour le taux de survie.
  - SciPy   : test de Shapiro (normalité), test t de Welch (comparaison de moyennes),
              corrélation de Pearson, mesures de forme (skew, kurtosis).
  - Matplotlib / Seaborn : séries temporelles, histogrammes, barplots.
""")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 8462: invalid continuation byte